In [1]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

# 1. Define the model name
model_name = "distilbert-base-cased"

# 2. Download/Load the Tokenizer (essential for converting text to IDs)
tokenizer = DistilBertTokenizer.from_pretrained(model_name)

# 3. Download/Load the Model with a classification head
# num_labels=2 for your Human vs. Bot binary task
model = DistilBertForSequenceClassification.from_pretrained(model_name, num_labels=2)

c:\Users\Wei Yong\Documents\GitHub\universal-deception-detector\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 5804.62it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- 

In [2]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path('datasets')

finance_df = pd.read_json(DATA_DIR / 'finance.jsonl', lines=True)
medicine_df = pd.read_json(DATA_DIR / 'medicine.jsonl', lines=True)
open_qa_df = pd.read_json(DATA_DIR / 'open_qa.jsonl', lines=True)
reddit_eli5_df = pd.read_json(DATA_DIR / 'reddit_eli5.jsonl', lines=True)
wiki_csai_df = pd.read_json(DATA_DIR / 'wiki_csai.jsonl', lines=True)

In [ ]:
dataframes = {
    "finance": finance_df.copy(),
    "medicine": medicine_df.copy(),
    "open_qa": open_qa_df.copy(),
    "reddit_eli5": reddit_eli5_df.copy(),
    "wiki_csai": wiki_csai_df.copy(),
}

text_candidates = ["text", "content", "body", "sentence", "input", "prompt", "question", "response", "answer"]
label_candidates = ["label", "target", "class", "is_ai", "is_human", "source"]
human_candidates = ["human_answers", "human_answer", "human", "reference", "gold", "real_answer"]
ai_candidates = ["ai_answers", "ai_answer", "chatgpt_answers", "chatgpt_answer", "generated_answer", "bot_answer", "model_answer"]


def first_match(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None


def to_clean_text(x):
    if isinstance(x, (list, tuple, set)):
        parts = [str(v).strip() for v in x if str(v).strip()]
        return " ".join(parts) if parts else None

    # Handle numpy/pandas array-likes without calling pd.isna on the whole array.
    if hasattr(x, "tolist") and not isinstance(x, (str, bytes, bytearray)):
        arr = x.tolist()
        if isinstance(arr, list):
            parts = [str(v).strip() for v in arr if str(v).strip()]
            return " ".join(parts) if parts else None

    if x is None:
        return None

    try:
        if pd.isna(x):
            return None
    except Exception:
        pass

    s = str(x).strip()
    return s if s else None


def normalize_label(x):
    if pd.isna(x):
        return None
    s = str(x).strip().lower()
    if s in ["0", "human", "real", "false", "non-ai", "non_ai", "not_ai"]:
        return 0
    if s in ["1", "ai", "bot", "generated", "true", "synthetic", "machine"]:
        return 1
    try:
        n = int(float(s))
        return n if n in [0, 1] else None
    except Exception:
        return None


def build_from_text_label(df):
    text_col = first_match(df.columns, text_candidates)
    label_col = first_match(df.columns, label_candidates)
    if text_col is None or label_col is None:
        return None

    out = df[[text_col, label_col]].copy()
    out.columns = ["text", "label"]
    out["text"] = out["text"].apply(to_clean_text)
    out["label"] = out["label"].apply(normalize_label)
    out = out.dropna(subset=["text", "label"])
    out = out[out["label"].isin([0, 1])].copy()
    out["label"] = out["label"].astype(int)
    return out.reset_index(drop=True)


def build_from_human_ai_pairs(df):
    human_col = first_match(df.columns, human_candidates)
    ai_col = first_match(df.columns, ai_candidates)

    if human_col is None:
        for c in df.columns:
            lc = str(c).lower()
            if "human" in lc:
                human_col = c
                break

    if ai_col is None:
        for c in df.columns:
            lc = str(c).lower()
            if ("ai" in lc) or ("gpt" in lc) or ("bot" in lc) or ("generated" in lc) or ("model" in lc):
                if c != human_col:
                    ai_col = c
                    break

    if human_col is None or ai_col is None:
        return None

    human_text = df[human_col].apply(to_clean_text)
    ai_text = df[ai_col].apply(to_clean_text)

    human_df = pd.DataFrame({"text": human_text, "label": 0})
    ai_df = pd.DataFrame({"text": ai_text, "label": 1})

    out = pd.concat([human_df, ai_df], ignore_index=True)
    out = out.dropna(subset=["text"])
    out = out[out["text"].str.len() > 0].copy()
    out["label"] = out["label"].astype(int)
    return out.reset_index(drop=True)


prepared = {}
for name, df in dataframes.items():
    out = build_from_text_label(df)
    if out is None:
        out = build_from_human_ai_pairs(df)

    if out is None:
        raise ValueError(f"{name}: could not infer text/label schema. Found columns: {list(df.columns)}")

    prepared[name] = out
    print(f"{name}: rows={len(out)}, label_0={(out['label'] == 0).sum()}, label_1={(out['label'] == 1).sum()}")

finance: rows=7866, label_0=3933, label_1=3933
medicine: rows=2494, label_0=1248, label_1=1246
open_qa: rows=2374, label_0=1187, label_1=1187
reddit_eli5: rows=33769, label_0=17112, label_1=16657
wiki_csai: rows=1684, label_0=842, label_1=842


In [8]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "accelerate>=1.1.0", "-q"])
print("accelerate installed successfully")

accelerate installed successfully


In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification


class TextClassificationDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256):
        self.texts = df["text"].tolist()
        self.labels = df["label"].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


def evaluate_model(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in dataloader:
            labels = batch["labels"].to(device)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1)
            
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
    
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_names = list(prepared.keys())
all_results = []

for test_name in all_names:
    print(f"\n{'='*60}")
    print(f"Training with {test_name} held out")
    print(f"{'='*60}")
    
    train_names = [n for n in all_names if n != test_name]
    train_df = pd.concat([prepared[n] for n in train_names], ignore_index=True)
    test_df = prepared[test_name].reset_index(drop=True)

    model_name = "distilbert-base-cased"
    tokenizer = DistilBertTokenizer.from_pretrained(model_name)
    model = DistilBertForSequenceClassification.from_pretrained(model_name, num_labels=2)
    model = model.to(device)

    train_ds = TextClassificationDataset(train_df, tokenizer, max_length=256)
    test_ds = TextClassificationDataset(test_df, tokenizer, max_length=256)
    
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    num_epochs = 2
    
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for step, batch in enumerate(train_loader):
            labels = batch["labels"].to(device)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            
            total_loss += loss.item()
            
            if (step + 1) % 50 == 0:
                print(f"Epoch {epoch+1}/{num_epochs}, Step {step+1}, Loss: {loss.item():.4f}")
        
        print(f"Epoch {epoch+1} completed. Average Loss: {total_loss / len(train_loader):.4f}")
    
    metrics = evaluate_model(model, test_loader, device)

    row = {
        "held_out": test_name,
        "accuracy": metrics["accuracy"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
    }
    all_results.append(row)

    print(
        f"{test_name}: "
        f"accuracy={row['accuracy']:.4f}, "
        f"precision={row['precision']:.4f}, "
        f"recall={row['recall']:.4f}, "
        f"f1={row['f1']:.4f}"
    )

results_df = pd.DataFrame(all_results).sort_values("held_out").reset_index(drop=True)
print("\n" + "="*60)
print("Per-dataset metrics:")
print("="*60)
print(results_df)
print("\nAverage metrics:")
print(results_df[["accuracy", "precision", "recall", "f1"]].mean())


Training with finance held out


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 5155.81it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/2, Step 50, Loss: 0.3784
Epoch 1/2, Step 100, Loss: 0.0208
Epoch 1/2, Step 150, Loss: 0.0371
Epoch 1/2, Step 200, Loss: 0.0892
Epoch 1/2, Step 250, Loss: 0.0047
Epoch 1/2, Step 300, Loss: 0.0081
